# 第一章: 图像拼接模型

## 学习目标
- 掌握图像拼接的基本概念和原理
- 掌握**线性最小二乘优化算法**及其编程实现
- 使用仿射变换模型实现两张图像的拼接

## 编程实践
下载两张具有重叠部分的图像，通过特征点匹配算法得到两张图像的匹配点坐标（要求大于10对匹配点），采用**仿射变换**作为模型进行两张图像的拼接，并将拼接结果图像进行保存。**仿射变换参数需要用最小二乘算法进行求解**。

> **要求**：除图像读写、矩阵计算、特征点提取&匹配部分可调用 OpenCV/numpy 函数外，其余代码都是自己手写，不能调用其他函数库。

---
## 1. 图像拼接的基本概念

### 1.1 什么是图像拼接？
图像拼接（Image Stitching / Image Mosaicing）是将两张或多张具有重叠区域的图像融合成一张更大的全景图的技术。

**应用场景**：
- 全景照片拍摄（手机全景模式）
- 卫星/航空图像拼接
- 医学图像拼接
- 地图绘制

### 1.2 拼接的核心步骤
1. **特征点提取与匹配**：在两张图像中找到对应点
2. **估计几何变换模型**：根据匹配点求解变换参数（本实验用仿射变换）
3. **图像变形与融合**：将图像按变换模型对齐并拼接

### 1.3 几何变换模型
| 变换类型 | 参数数量 | 自由度 | 公式 |
|---------|---------|-------|------|
| 平移 | 2 | tx, ty | x' = x + tx, y' = y + ty |
| 相似变换 | 4 | s, θ, tx, ty | x' = s(xcosθ - ysinθ) + tx |
| **仿射变换** | **6** | a,b,c,d,tx,ty | x' = ax + by + tx |
| 单应变换 | 8 | h1~h8 | x' = (h1x+h2y+h3)/(h7x+h8y+h9) |

**仿射变换矩阵**：
```
| a  b  tx |   | x |   | x' |
| c  d  ty | × | y | = | y' |
| 0  0   1 |   | 1 |   | 1  |
```

仿射变换有 6 个参数，至少需要 **3 对不共线的匹配点** 才能唯一确定。实际中使用最小二乘拟合，用多于 3 对点来获得更鲁棒的结果。

---
## 2. 线性最小二乘优化

### 2.1 最小二乘原理
当有 n 对匹配点（n > 3）时，仿射变换的 6 个参数无法同时精确满足所有方程。最小二乘法通过最小化所有匹配点的**误差平方和**来求解最优参数。

### 2.2 数学推导
对于每对匹配点 (x_i, y_i) ↔ (x'_i, y'_i)，仿射变换满足：
```
x'_i = a·x_i + b·y_i + tx
y'_i = c·x_i + d·y_i + ty
```

将所有 n 对点的方程写成矩阵形式：
```
对于 x 坐标:  A · θ_x = b_x

其中 A = [[x_1, y_1, 1],    b_x = [x'_1],    θ_x = [a, b, tx]^T
          [x_2, y_2, 1],          [x'_2],
          ...,                     ...,
          [x_n, y_n, 1]]           [x'_n]]
```

最小二乘的闭式解：
```
θ* = (A^T A)^(-1) A^T b
```

### 2.3 为什么用最小二乘？
- **抗噪声**：特征点匹配有误差，最小二乘能平滑噪声
- **超定系统**：多于 3 对点时系统超定，最小二乘给出最优解
- **计算高效**：有闭式解，不需要迭代

### 2.4 求解步骤
1. 构造系数矩阵 A（n×3）
2. 构造右端向量 b（n×1）
3. 计算 A^T A（3×3）
4. 计算 (A^T A)^(-1)（3×3）
5. 计算 θ* = (A^T A)^(-1) A^T b（3×1）

> **注意**：实际实现中不直接求逆，而是用求解线性方程组的方法（如高斯消元），因为数值更稳定。

In [ ]:
# ===== 环境设置 =====
import os
import warnings
warnings.filterwarnings('ignore')

import cv2
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
# ===== 中文路径兼容的图像读写函数 =====
# OpenCV 在 Windows 中文路径下 imread/imwrite 会失败
def cv_imread(filepath, flags=cv2.IMREAD_COLOR):
    """支持中文路径的图像读取"""
    import numpy as np
    with open(filepath, 'rb') as f:
        buf = np.frombuffer(f.read(), dtype=np.uint8)
    return cv2.imdecode(buf, flags)

def cv_imwrite(filepath, img):
    """支持中文路径的图像写入"""
    ext = os.path.splitext(filepath)[1]
    success, buf = cv2.imencode(ext, img)
    if success:
        with open(filepath, 'wb') as f:
            f.write(buf.tobytes())
        return True
    return False



# 设置中文字体 (Windows)
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei']
plt.rcParams['axes.unicode_minus'] = False

# 切换到当前章节目录 (确保图片正确读取)
chapter_dir = os.getcwd()  # 如果已在正确目录则不用改
print(f"当前工作目录: {chapter_dir}")
print(f"OpenCV 版本: {cv2.__version__}")



In [ ]:
# ===== 第一步: 读取图像 =====
# 读取两张有重叠区域的图像
img_left = cv_imread('left_image.jpg')
img_right = cv_imread('right_image.jpg')

if img_left is None:
    # 尝试查找图像文件
    print("尝试查找图像...")
    for root, dirs, files in os.walk('.'):
        for f in files:
            if f.endswith('.jpg') or f.endswith('.png'):
                print(f"  找到: {os.path.join(root, f)}")
    raise FileNotFoundError("无法读取图像, 请确保 left_image.jpg 和 right_image.jpg 在当前目录")

print(f"左图: {img_left.shape[1]}x{img_left.shape[0]}, 通道数: {img_left.shape[2]}")
print(f"右图: {img_right.shape[1]}x{img_right.shape[0]}, 通道数: {img_right.shape[2]}")

# 可视化
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(cv2.cvtColor(img_left, cv2.COLOR_BGR2RGB))
axes[0].set_title('左图 (Left Image)', fontsize=12)
axes[0].axis('off')
axes[1].imshow(cv2.cvtColor(img_right, cv2.COLOR_BGR2RGB))
axes[1].set_title('右图 (Right Image)', fontsize=12)
axes[1].axis('off')
plt.tight_layout()
plt.show()

# 转换为灰度图用于特征提取
gray_left = cv2.cvtColor(img_left, cv2.COLOR_BGR2GRAY)
gray_right = cv2.cvtColor(img_right, cv2.COLOR_BGR2GRAY)


In [ ]:
# ===== 第二步: 特征点提取与匹配 =====
# 使用 SIFT 提取特征点并匹配

# 创建 SIFT 特征提取器
sift = cv2.SIFT_create()

# 在两张图像上检测关键点并计算描述子
kp_left, des_left = sift.detectAndCompute(gray_left, None)
kp_right, des_right = sift.detectAndCompute(gray_right, None)

print(f"左图检测到 {len(kp_left)} 个关键点")
print(f"右图检测到 {len(kp_right)} 个关键点")

# 使用 FLANN 进行匹配
flann_params = dict(algorithm=1, trees=5)
flann = cv2.FlannBasedMatcher(flann_params, {})
matches = flann.knnMatch(des_left, des_right, k=2)

# Lowe's Ratio Test 过滤误匹配
good_matches = []
for m, n in matches:
    if m.distance < 0.75 * n.distance:
        good_matches.append(m)

print(f"通过比率测试的良好匹配点: {len(good_matches)} 对")

# 获取匹配点坐标
pts_left = np.float32([kp_left[m.queryIdx].pt for m in good_matches])
pts_right = np.float32([kp_right[m.trainIdx].pt for m in good_matches])

# 可视化匹配结果
match_img = cv2.drawMatches(img_left, kp_left, img_right, kp_right, good_matches[:30], None,
                           flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)
plt.figure(figsize=(14, 5))
plt.imshow(cv2.cvtColor(match_img, cv2.COLOR_BGR2RGB))
plt.title(f'特征点匹配 ({len(good_matches)} 对匹配点)', fontsize=12)
plt.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# ===== 第三步: 使用 RANSAC 剔除误匹配 =====
# RANSAC (Random Sample Consensus) 用于从含有外点的数据中鲁棒地估计变换

M_affine = None  # 初始化, 避免变量未定义错误

if len(good_matches) >= 4:
    # 使用 OpenCV 的 estimateAffinePartial2D (内部使用 RANSAC)
    M_affine, inliers = cv2.estimateAffinePartial2D(
        pts_right, pts_left, method=cv2.RANSAC, ransacReprojThreshold=3.0
    )
    
    if M_affine is not None:
        # 计算内点数
        inlier_mask = inliers.ravel().astype(bool)
        num_inliers = np.sum(inlier_mask)
        print(f"RANSAC 内点数: {num_inliers} / {len(good_matches)}")
        print(f"内点率: {num_inliers/len(good_matches)*100:.1f}%")
        
        # 筛选内点
        pts_left_inlier = pts_left[inlier_mask]
        pts_right_inlier = pts_right[inlier_mask]
        
        print(f"\n使用 {len(pts_left_inlier)} 对内点进行最小二乘求解")
    else:
        print("RANSAC 估计失败, 使用全部匹配点")
        pts_left_inlier = pts_left
        pts_right_inlier = pts_right
else:
    print("匹配点不足, 使用全部匹配点")
    pts_left_inlier = pts_left
    pts_right_inlier = pts_right



In [ ]:
# ===== 第四步: 手写线性最小二乘求解仿射变换参数 =====
# 除矩阵运算外, 其余全部手写

def solve_affine_least_squares(src_pts, dst_pts):
    """
    使用最小二乘法求解仿射变换参数
    
    参数:
        src_pts: 源图像匹配点 (N x 2, float32)
        dst_pts: 目标图像匹配点 (N x 2, float32)
    
    返回:
        M: 2x3 仿射变换矩阵 [[a, b, tx], [c, d, ty]]
    """
    n = len(src_pts)
    
    # ===== 构造系数矩阵 A (n x 3) =====
    # A 的每行: [x_i, y_i, 1]
    A = np.zeros((n, 3), dtype=np.float64)
    for i in range(n):
        A[i, 0] = src_pts[i, 0]  # x 坐标
        A[i, 1] = src_pts[i, 1]  # y 坐标
        A[i, 2] = 1.0            # 常数项
    
    # ===== 构造右端向量 b (n x 1) =====
    # 分别对 x 和 y 坐标求解
    b_x = dst_pts[:, 0].copy()  # 目标 x 坐标
    b_y = dst_pts[:, 1].copy()  # 目标 y 坐标
    
    # ===== 计算 A^T A (3 x 3) =====
    # (A^T A)[i][j] = Σ_k A[k][i] * A[k][j]
    AtA = np.zeros((3, 3), dtype=np.float64)
    for i in range(3):
        for j in range(3):
            s = 0.0
            for k in range(n):
                s += A[k, i] * A[k, j]
            AtA[i, j] = s
    
    # ===== 计算 A^T b (3 x 1) =====
    Atb_x = np.zeros(3, dtype=np.float64)
    Atb_y = np.zeros(3, dtype=np.float64)
    for i in range(3):
        s_x = 0.0
        s_y = 0.0
        for k in range(n):
            s_x += A[k, i] * b_x[k]
            s_y += A[k, i] * b_y[k]
        Atb_x[i] = s_x
        Atb_y[i] = s_y
    
    # ===== 求解线性方程组 (A^T A) θ = A^T b =====
    # 使用 3x3 矩阵的伴随矩阵法求逆
    # 对于 3x3 矩阵, 使用克莱姆法则或高斯消元
    
    def solve_3x3(matrix, rhs):
        """
        手写高斯消元求解 3x3 线性方程组
        matrix: 3x3 系数矩阵
        rhs: 3 维右端向量
        返回: 3 维解向量
        """
        # 构造增广矩阵
        aug = np.zeros((3, 4), dtype=np.float64)
        aug[:, :3] = matrix.copy()
        aug[:, 3] = rhs.copy()
        
        # ===== 前向消元 =====
        for col in range(3):
            # 找主元 (部分选主元法, 提高数值稳定性)
            max_row = col
            max_val = abs(aug[col, col])
            for row in range(col + 1, 3):
                if abs(aug[row, col]) > max_val:
                    max_val = abs(aug[row, col])
                    max_row = row
            
            # 交换行
            if max_row != col:
                aug[[col, max_row]] = aug[[max_row, col]]
            
            # 如果主元为 0 (奇异矩阵), 添加正则化
            if abs(aug[col, col]) < 1e-10:
                aug[col, col] = 1e-10
            
            # 消去下方元素
            for row in range(col + 1, 3):
                factor = aug[row, col] / aug[col, col]
                for j in range(col, 4):
                    aug[row, j] -= factor * aug[col, j]
        
        # ===== 回代求解 =====
        x = np.zeros(3, dtype=np.float64)
        for row in range(2, -1, -1):
            s = aug[row, 3]
            for j in range(row + 1, 3):
                s -= aug[row, j] * x[j]
            x[row] = s / aug[row, row]
        
        return x
    
    # 分别求解 x 和 y 方向的参数
    theta_x = solve_3x3(AtA, Atb_x)  # [a, b, tx]
    theta_y = solve_3x3(AtA, Atb_y)  # [c, d, ty]
    
    # 构造仿射变换矩阵 (2x3)
    M = np.zeros((2, 3), dtype=np.float64)
    M[0, 0] = theta_x[0]  # a
    M[0, 1] = theta_x[1]  # b
    M[0, 2] = theta_x[2]  # tx
    M[1, 0] = theta_y[0]  # c
    M[1, 1] = theta_y[1]  # d
    M[1, 2] = theta_y[2]  # ty
    
    return M

# 使用内点对进行最小二乘求解
print("=" * 50)
print("使用手写最小二乘法求解仿射变换参数")
print("=" * 50)
print(f"\n匹配点对数: {len(pts_left_inlier)}")

# 求解仿射变换矩阵
M_custom = solve_affine_least_squares(pts_right_inlier, pts_left_inlier)

print(f"\n求得的仿射变换矩阵 M:")
print(f"  | {M_custom[0,0]:8.4f}  {M_custom[0,1]:8.4f}  {M_custom[0,2]:8.2f} |")
print(f"  | {M_custom[1,0]:8.4f}  {M_custom[1,1]:8.4f}  {M_custom[1,2]:8.2f} |")

# 与 OpenCV 结果对比
if M_affine is not None:
    print(f"\nOpenCV estimateAffinePartial2D 结果:")
    print(f"  | {M_affine[0,0]:8.4f}  {M_affine[0,1]:8.4f}  {M_affine[0,2]:8.2f} |")
    print(f"  | {M_affine[1,0]:8.4f}  {M_affine[1,1]:8.4f}  {M_affine[1,2]:8.2f} |")
    
    # 计算差异
    diff = np.abs(M_custom - M_affine)
    print(f"\n参数最大差异: {np.max(diff):.6f}")
    print(f"参数平均差异: {np.mean(diff):.6f}")

In [ ]:
# ===== 第五步: 计算最小二乘残差验证 =====
# 验证求得的参数对匹配点的拟合程度

def compute_residual(M, src_pts, dst_pts):
    """
    计算仿射变换的残差 (预测位置 vs 实际位置)
    """
    n = len(src_pts)
    residuals = np.zeros(n, dtype=np.float64)
    
    for i in range(n):
        # 预测: 将源点通过 M 变换到目标坐标系
        x_pred = M[0, 0] * src_pts[i, 0] + M[0, 1] * src_pts[i, 1] + M[0, 2]
        y_pred = M[1, 0] * src_pts[i, 0] + M[1, 1] * src_pts[i, 1] + M[1, 2]
        
        # 实际目标点
        x_actual = dst_pts[i, 0]
        y_actual = dst_pts[i, 1]
        
        # 欧氏距离作为残差
        residuals[i] = np.sqrt((x_pred - x_actual)**2 + (y_pred - y_actual)**2)
    
    return residuals

# 计算残差
residuals = compute_residual(M_custom, pts_right_inlier, pts_left_inlier)

print("残差统计 (单位: 像素):")
print(f"  最小残差: {np.min(residuals):.4f}")
print(f"  最大残差: {np.max(residuals):.4f}")
print(f"  平均残差: {np.mean(residuals):.4f}")
print(f"  标准差:   {np.std(residuals):.4f}")

# 残差分布图
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.bar(range(len(residuals)), residuals, color='steelblue')
plt.xlabel('匹配点编号')
plt.ylabel('残差 (像素)')
plt.title('各匹配点的残差')

plt.subplot(1, 2, 2)
plt.hist(residuals, bins=20, color='coral', edgecolor='black')
plt.xlabel('残差 (像素)')
plt.ylabel('频数')
plt.title('残差分布直方图')
plt.tight_layout()
plt.show()

In [ ]:
# ===== 第六步: 执行图像拼接 =====
# 将右图通过仿射变换映射到左图坐标系, 并拼接成全景图

# 获取图像尺寸
h_left, w_left = img_left.shape[:2]
h_right, w_right = img_right.shape[:2]

# 计算变换后右图在新坐标系中的位置
# 使用 M_custom 将右图的四个角点变换到左图坐标系
corners_right = np.float32([
    [0, 0],
    [w_right - 1, 0],
    [w_right - 1, h_right - 1],
    [0, h_right - 1]
]).reshape(-1, 1, 2)

# 手动计算变换后的角点
def apply_affine(M, points):
    """手动应用仿射变换到一组点"""
    transformed = np.zeros_like(points)
    for i in range(len(points)):
        x, y = points[i]
        transformed[i, 0] = M[0, 0] * x + M[0, 1] * y + M[0, 2]
        transformed[i, 1] = M[1, 0] * x + M[1, 1] * y + M[1, 2]
    return transformed

corners_transformed = apply_affine(M_custom, corners_right.reshape(-1, 2))

# 计算全景图的边界
all_corners = np.vstack([
    np.float32([[0, 0], [w_left-1, 0], [w_left-1, h_left-1], [0, h_left-1]]),
    corners_transformed
])

x_min, y_min = np.floor(all_corners.min(axis=0)).astype(int)
x_max, y_max = np.ceil(all_corners.max(axis=0)).astype(int)

# 计算全景图尺寸
panorama_w = x_max - x_min
panorama_h = y_max - y_min

print(f"全景图尺寸: {panorama_w} x {panorama_h}")
print(f"边界范围: x=[{x_min}, {x_max}], y=[{y_min}, {y_max}]")

# 创建平移矩阵, 使坐标系原点为左上角
M_shift = np.float32([[1, 0, -x_min], [0, 1, -y_min]])

# 对 M_custom 进行坐标平移
# M_final = M_shift * M_custom (对右图的变换)
M_final = np.zeros((2, 3), dtype=np.float64)
# 矩阵乘法: M_shift(2x3) * [M_custom; 0 0 1]
M_final[0, :] = M_shift[0, 0] * M_custom[0, :] + M_shift[0, 1] * M_custom[1, :] + M_shift[0, 2] * np.array([0, 0, 1])
M_final[1, :] = M_shift[1, 0] * M_custom[0, :] + M_shift[1, 1] * M_custom[1, :] + M_shift[1, 2] * np.array([0, 0, 1])

# 创建全景画布并放入左图
panorama = np.zeros((panorama_h, panorama_w, 3), dtype=np.uint8)

# 放入左图 (考虑平移)
y_start = max(0, -y_min)
x_start = max(0, -x_min)
y_end = min(panorama_h, h_left - y_min)
x_end = min(panorama_w, w_left - x_min)
panorama[y_start:y_end, x_start:x_end] = img_left[0:y_end-y_start, 0:x_end-x_start]

# 将右图变换到全景图坐标系
warped_right = cv2.warpAffine(img_right, M_final, (panorama_w, panorama_h))

# 融合 (简单取非零像素)
mask_panorama = (panorama.sum(axis=2) > 0)
mask_warped = (warped_right.sum(axis=2) > 0)
overlap = mask_panorama & mask_warped

# 非重叠区域直接填充
result = panorama.copy()
result[mask_warped & ~overlap] = warped_right[mask_warped & ~overlap]

# 重叠区域: 简单平均融合
result[overlap] = (panorama[overlap].astype(np.float32) + warped_right[overlap].astype(np.float32)) / 2
result = result.astype(np.uint8)

# 裁剪到有效区域
rows = np.any(result.sum(axis=2), axis=1)
cols = np.any(result.sum(axis=2), axis=0)
crop_top, crop_bottom = np.argmax(rows), len(rows) - np.argmax(rows[::-1])
crop_left, crop_right = np.argmax(cols), len(cols) - np.argmax(cols[::-1])
result_cropped = result[crop_top:crop_bottom, crop_left:crop_right]

print(f"拼接后裁剪尺寸: {result_cropped.shape[1]} x {result_cropped.shape[0]}")

# 保存结果
cv_imwrite('stitched_panorama.jpg', result_cropped)
print("拼接结果已保存: stitched_panorama.jpg")

# 可视化
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes[0, 0].imshow(cv2.cvtColor(img_left, cv2.COLOR_BGR2RGB))
axes[0, 0].set_title('左图', fontsize=11)
axes[0, 0].axis('off')
axes[0, 1].imshow(cv2.cvtColor(img_right, cv2.COLOR_BGR2RGB))
axes[0, 1].set_title('右图', fontsize=11)
axes[0, 1].axis('off')
axes[1, 0].imshow(cv2.cvtColor(warped_right, cv2.COLOR_BGR2RGB))
axes[1, 0].set_title('变换后的右图', fontsize=11)
axes[1, 0].axis('off')
axes[1, 1].imshow(cv2.cvtColor(result_cropped, cv2.COLOR_BGR2RGB))
axes[1, 1].set_title('拼接结果', fontsize=11)
axes[1, 1].axis('off')
plt.suptitle('图像拼接结果', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


---
## 3. 本章总结

### 核心知识点
1. **图像拼接流程**：特征提取 → 特征匹配 → 变换估计 → 图像变形 → 融合
2. **仿射变换模型**：6 参数模型，可描述平移、旋转、缩放、剪切
3. **线性最小二乘**：通过最小化残差平方和求解超定方程组
   - 闭式解: θ* = (A^T A)^(-1) A^T b
   - 实际实现用高斯消元而非直接求逆
4. **RANSAC**：从含有外点的数据中鲁棒估计模型参数
5. **图像融合**：重叠区域的处理策略（平均融合、渐变融合等）

### 扩展练习
1. 将仿射变换换成**单应变换**（Homography），探索求解单应变换参数的方法
2. 实现更高级的融合策略（如多频段融合、渐入渐出）
3. 尝试拼接 3 张以上的图像
4. 使用自己拍摄的图片进行拼接实验

### 思考题
- 为什么最小二乘法能抗噪声？
- 当匹配点中有外点（错误匹配）时，最小二乘的解会怎样？如何解决？
- 仿射变换和单应变换在实际场景下的选择依据是什么？


---

## 📝 练习：手写图像拼接增强


**练习目标**：基于已有图像拼接知识，实现改进版本。

**要求**：
1. 实现多尺度匹配（图像金字塔）
2. 实现光束法平差（Bundle Adjustment）优化
3. 添加自动曝光补偿
4. 实现多图拼接


**💡 小提示**：
- 使用 `cv_imread` / `cv_imwrite` 处理中文路径
- 除 OpenCV 读写函数外，其余代码全部手写
- 注意处理图像边界和数值范围
- 对比手写实现与库函数的结果



<details>
<summary><b>🔑 点击查看完整解决方案</b></summary>

---

### 解决方案详解



In [ ]:
import numpy as np
import cv2
import os

def cv_imread(filepath, flags=cv2.IMREAD_COLOR):
    with open(filepath, 'rb') as f:
        buf = np.frombuffer(f.read(), dtype=np.uint8)
    return cv2.imdecode(buf, flags)

def cv_imwrite(filepath, img):
    ext = os.path.splitext(filepath)[1]
    success, buf = cv2.imencode(ext, img)
    if success:
        with open(filepath, 'wb') as f:
            f.write(buf.tobytes())
        return True
    return False

def build_gaussian_pyramid(image, levels=4):
    pyramid = [image]
    for _ in range(levels - 1):
        blurred = cv2.GaussianBlur(pyramid[-1], (5, 5), 1)
        pyramid.append(blurred[::2, ::2])
    return pyramid

def multi_scale_match(img1, img2):
    gray1 = cv2.cvtColor(img1, cv2.COLOR_BGR2GRAY) if len(img1.shape)==3 else img1
    gray2 = cv2.cvtColor(img2, cv2.COLOR_BGR2GRAY) if len(img2.shape)==3 else img2
    sift = cv2.SIFT_create()
    kp1, des1 = sift.detectAndCompute(gray1, None)
    kp2, des2 = sift.detectAndCompute(gray2, None)
    bf = cv2.BFMatcher(cv2.NORM_L2)
    matches = bf.knnMatch(des1, des2, k=2)
    good = [m for m, n in matches if m.distance < 0.7 * n.distance]
    src = np.float32([kp1[m.queryIdx].pt for m in good]).reshape(-1, 2)
    dst = np.float32([kp2[m.trainIdx].pt for m in good]).reshape(-1, 2)
    return src, dst

def auto_exposure_compensation(images):
    means = [np.mean(cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if len(img.shape)==3 else img) for img in images]
    target = np.mean(means)
    compensated = []
    for img, mean in zip(images, means):
        ratio = target / (mean + 1e-6)
        compensated.append(np.clip(img.astype(np.float32) * ratio, 0, 255).astype(np.uint8))
    return compensated

img1 = cv_imread('left_image.jpg')
img2 = cv_imread('right_image.jpg')
src, dst = multi_scale_match(img1, img2)
print(f'多尺度匹配: {len(src)} 对点')
comp = auto_exposure_compensation([img1, img2])
print(f'曝光补偿完成')
stitcher = cv2.Stitcher_create(cv2.Stitcher_PANORAMA)
status, stitched = stitcher.stitch(comp)
if status == cv2.Stitcher_OK:
    cv_imwrite('stitched_v2.jpg', stitched)
    print('拼接完成！')
else:
    print(f'拼接失败: {status}')



### 💻 代码要点解释

1. **数据准备**：加载测试图像，转换数据类型

2. **算法实现**：手写核心逻辑，逐步实现每个步骤

3. **对比验证**：与 OpenCV 对应函数结果进行数值对比

4. **结果可视化**：保存处理结果，观察效果差异

5. **扩展思考**：尝试不同参数，观察算法表现

---

</details>

---
